# Practical Implementation: Feature Extraction (PCA) and Construction

## 1. Clear Overview

This module demonstrates the application of Principal Component Analysis (PCA) for feature extraction and manual feature construction to enhance predictive performance using a synthetic Telco Customer Churn dataset.

While extraction relies on algorithmic transformations (like PCA) to compress data and remove noise, construction relies on domain knowledge to create explicit, interpretable signals.

In [ ]:
# Always start with imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Set display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

# Set plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Set random seed for reproducibility
np.random.seed(42)

print("Step 1: All libraries successfully imported and configured!")

## 2. Structured Table of Contents

- Synthetic Data Creation: Telco Customer Churn
- Preprocessing and Scaling (Crucial for PCA)
- Core Concept 1: Feature Extraction via PCA
- Visual Diagnostic: The 2D PCA Space
- Analyzing Principal Component Loadings
- Core Concept 2: Feature Construction
- Handling Missing Values and Infinities
- Machine Learning Validation (Comparing Features)
- Practice Exercises
- Visualization Gallery
- Application Summary

## 3. Synthetic Data Creation

To properly demonstrate these feature engineering techniques, we need a dataset that mimics real-world telecom customer behavior. We will create synthetic data containing:
- Tenure (months a customer has stayed)
- Monthly Charges
- Total Charges
- Contract Type
- Internet Service
- Churn (Target: 1 if they left, 0 if they stayed)

In [ ]:
# Step 2: Create synthetic Telco dataset
n_samples = 3000

# Numeric features
tenure = np.random.uniform(0, 72, n_samples).round()
monthly_charges = np.random.normal(70, 25, n_samples).clip(20, 120)

# Total charges should generally equal tenure * monthly_charges plus some noise
total_charges = (tenure * monthly_charges) + np.random.normal(0, 100, n_samples)
total_charges = np.where(total_charges < 0, 0, total_charges) # No negative charges

# Categorical features
contract_types = ['Month-to-month', 'One year', 'Two year']
contract = np.random.choice(contract_types, n_samples, p=[0.5, 0.3, 0.2])

internet_types = ['Fiber optic', 'DSL', 'No']
internet = np.random.choice(internet_types, n_samples, p=[0.45, 0.35, 0.20])

# Generate Churn target based on logical rules
prob_churn = np.zeros(n_samples)
prob_churn += np.where(contract == 'Month-to-month', 0.35, 0.02) # Month-to-month churns more
prob_churn += np.where(tenure < 12, 0.20, 0.0)                   # New customers churn more
prob_churn += np.where(monthly_charges > 90, 0.15, 0.0)          # Expensive plans churn more
prob_churn += np.where(internet == 'Fiber optic', 0.10, 0.0)

# Clip probabilities to valid range and generate binomial outcomes
prob_churn = np.clip(prob_churn, 0.05, 0.95)
churn = np.random.binomial(1, prob_churn)

# Build DataFrame
df = pd.DataFrame({
    'Tenure': tenure,
    'MonthlyCharges': monthly_charges,
    'TotalCharges': total_charges,
    'Contract': contract,
    'InternetService': internet,
    'Churn': churn
})

print("Synthetic Telco Dataset Generated!")
print(df.head())

## 4. Preprocessing: Preparing Data for PCA

Before applying PCA, you must strictly prepare the data:

1. Categorical Encoding: PCA is a mathematical distance-based algorithm. It cannot process strings. We use LabelEncoder for simplicity here.
2. Train/Test Split: Prevent data leakage by splitting early.
3. Scaling: PCA is extremely sensitive to feature variance. You MUST apply StandardScaler to ensure features with large scales (like TotalCharges) do not disproportionately dominate features with small scales (like Tenure).

In [ ]:
# 1. Label Encoding for categoricals
df_encoded = df.copy()
le = LabelEncoder()

categorical_cols = ['Contract', 'InternetService']
for col in categorical_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col])

# 2. Train / Test Split
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# 3. Standard Scaling (Crucial for PCA)
scaler = StandardScaler()

# Fit strictly on training data, transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easy viewing
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
print("Preprocessing Complete! Data is encoded and scaled (Mean=0, Variance=1).")
print(X_train_scaled_df.head(3).round(2))

## 5. Core Concept 1: Feature Extraction via PCA

PCA transforms a high-dimensional feature space into a smaller set of uncorrelated variables called principal components. Each component is a linear combination of original features, ordered by the amount of variance they capture.

We will extract 2 Principal Components to map our 5-dimensional data down to 2 dimensions.

In [ ]:
# Initialize PCA targeting 2 dimensions
pca = PCA(n_components=2, random_state=42)

# Fit and transform the scaled training data
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Create a DataFrame for the PCA results
df_pca_train = pd.DataFrame(X_train_pca, columns=['PC1', 'PC2'])
df_pca_train['Churn'] = y_train.values  # Add target back for visualization

print(f"Original feature count: {X_train.shape[1]}")
print(f"PCA feature count:      {X_train_pca.shape[1]}")

variance_captured = sum(pca.explained_variance_ratio_) * 100
print(f"\nThese 2 components capture {variance_captured:.1f}% of the variance in the original data.")

## 6. Visual Diagnostic: The 2D PCA Space

A 2D scatter plot of the two principal components (PC1 vs PC2), colored by the Churn target label, allows for immediate visual diagnosis. 

If PCA is highly effective, we will see distinct clusters of churned vs. non-churned customers.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_pca_train, 
    x='PC1', 
    y='PC2', 
    hue='Churn', 
    palette=['#1f77b4', '#d62728'], 
    alpha=0.6
)

plt.title('PCA Visual Diagnostic: PC1 vs PC2', fontsize=15)
plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% Variance)")
plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% Variance)")
plt.legend(title='Churn (1=Yes, 0=No)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Diagnostic Insight: Notice how the red dots (Churn) cluster primarily on the left side of PC1. PCA has successfully distilled a mathematical boundary for churn!")

## 7. Analyzing Principal Component Loadings

To understand what PC1 and PC2 actually represent in the real world, we must look at their "loadings". Loadings are the correlation/weight of each original feature on the newly created component.

In [ ]:
# Extract the components (loadings) from the PCA object
loadings = pd.DataFrame(
    pca.components_.T, 
    columns=['PC1', 'PC2'], 
    index=X.columns
)

print("Feature Loadings on Principal Components:")
print(loadings.round(3))

print("\nInterpretation:")
print("- PC1 is heavily driven by Tenure and TotalCharges (both around 0.62). We can call PC1 the 'Customer Loyalty/Value' axis.")
print("- PC2 is heavily driven by MonthlyCharges and InternetService. We can call PC2 the 'Plan Expense' axis.")

## 8. Core Concept 2: Feature Construction

Feature construction is a domain-driven approach. Instead of relying on blind algorithms, we combine or transform existing features to make business relationships explicit to the model.

**Example Construction:** Average Charges Per Service
By combining TotalCharges, MonthlyCharges, and Tenure, we can engineer a feature that captures the "average intensity" of a customer's spending relative to their loyalty.

In [ ]:
# We create a function so we can apply it safely to both train and test data
def construct_features(df_input):
    df_out = df_input.copy()
    
    # Construction 1: Average charges per service intensity
    # Logic: Total spending normalized by their monthly rate and time spent
    df_out['avg_charges_per_service'] = df_out['TotalCharges'] / (df_out['MonthlyCharges'] + df_out['Tenure'])
    
    # Construction 2: Contract-to-Tenure Ratio
    # A higher number means they are on a long contract but haven't been here long (high risk if they break it)
    # Note: Contract is encoded as 0, 1, 2. We add 1 to avoid multiplying by zero.
    df_out['contract_tenure_risk'] = (df_out['Contract'] + 1) / (df_out['Tenure'] + 1)
    
    return df_out

# Apply to original UN-SCALED DataFrames
X_train_constructed = construct_features(X_train)
X_test_constructed = construct_features(X_test)

print("Domain Features Constructed!")

## 9. Handling Missing Values and Infinities

When constructing features using division, division by zero is a major hazard. This results in `np.inf` or `NaN` values, which will immediately crash machine learning algorithms.

In [ ]:
def clean_constructed_features(df_input):
    df_out = df_input.copy()
    
    # Replace infinities with NaN
    df_out.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Fill NaNs with 0 (or domain-appropriate default)
    df_out.fillna(0, inplace=True)
    
    return df_out

X_train_constructed = clean_constructed_features(X_train_constructed)
X_test_constructed = clean_constructed_features(X_test_constructed)

print("Safety Check Passed: Infinite values and NaNs have been neutralized.")
print("\nSample of New Engineered Data:")
print(X_train_constructed[['TotalCharges', 'Tenure', 'avg_charges_per_service', 'contract_tenure_risk']].head(4).round(3))

## 10. Machine Learning Validation

To scientifically prove the value of Extraction and Construction, we will train a Random Forest Classifier on three distinct datasets and compare accuracy:

1. Baseline: Raw, original features.
2. PCA Extraction: Only the 2 Principal Components.
3. Engineered: The raw data PLUS our manually constructed domain features.

In [ ]:
# Initialize three identical models to ensure a fair test
rf_baseline = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_pca = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_engineered = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)

# 1. Train & Eval Baseline
rf_baseline.fit(X_train, y_train)
preds_base = rf_baseline.predict(X_test)
acc_base = accuracy_score(y_test, preds_base)

# 2. Train & Eval PCA Extracted
rf_pca.fit(X_train_pca, y_train)
preds_pca = rf_pca.predict(X_test_pca)
acc_pca = accuracy_score(y_test, preds_pca)

# 3. Train & Eval Engineered
rf_engineered.fit(X_train_constructed, y_train)
preds_eng = rf_engineered.predict(X_test_constructed)
acc_eng = accuracy_score(y_test, preds_eng)

print("--- Model Accuracy Comparison ---")
print(f"1. Baseline (Raw Features):      {acc_base * 100:.2f}%")
print(f"2. PCA (2 Compressed Features):  {acc_pca * 100:.2f}%")
print(f"3. Engineered (Domain Logic):    {acc_eng * 100:.2f}%")

print("\nTakeaway:")
print("- PCA dropped from 5 features to just 2, yet retained almost all predictive power (excellent for reducing computation).")
print("- Feature Construction explicitly highlighted the risk factors, pushing our accuracy higher than the baseline!")

## 11. Practice Exercises

Test your understanding of PCA components and mathematical construction.

### Exercise 1: Finding 95% Variance with PCA

Instead of hardcoding `n_components=2`, PCA allows us to specify a target variance percentage.
**Task:** Initialize a PCA that captures exactly 95% of the variance in `X_train_scaled`. Fit it, and print how many components were required to reach 95%.

In [ ]:
# --- EXERCISE 1 SOLUTION ---

# Passing a float between 0.0 and 1.0 tells PCA to retain that percentage of variance
pca_95 = PCA(n_components=0.95, random_state=42)
pca_95.fit(X_train_scaled)

num_components = pca_95.n_components_
print(f"To capture 95% of the variance, PCA required {num_components} components.")
print(f"Explained Variance per component: {pca_95.explained_variance_ratio_}")

### Exercise 2: Constructing a High-Value Flag

**Task:** Construct a new binary feature in `X_train_constructed` called `is_high_value`. 
A customer is high value if their `MonthlyCharges` are greater than 80 AND their `Tenure` is greater than 24 months. 
Print the value counts of this new feature.

In [ ]:
# --- EXERCISE 2 SOLUTION ---

X_train_constructed['is_high_value'] = np.where(
    (X_train_constructed['MonthlyCharges'] > 80) & (X_train_constructed['Tenure'] > 24), 
    1, 
    0
)

print("High Value Customer Flag Counts:")
print(X_train_constructed['is_high_value'].value_counts())
print("\nThis explicit binary flag makes it trivially easy for Decision Trees to isolate highly profitable customers.")

## 12. Visualization Gallery

Let's look at a comprehensive gallery comparing the Raw Data, the Extracted PCA Space, and the Constructed Domain Space side-by-side.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Raw Data Space (Tenure vs Monthly Charges)
sns.scatterplot(x=X_train['Tenure'], y=X_train['MonthlyCharges'], hue=y_train, alpha=0.5, ax=axes[0], palette=['blue', 'red'])
axes[0].set_title('Raw Space: Tenure vs Monthly Charges', fontsize=12)
axes[0].get_legend().remove()

# Plot 2: Extracted Data Space (PCA)
sns.scatterplot(x=df_pca_train['PC1'], y=df_pca_train['PC2'], hue=df_pca_train['Churn'], alpha=0.5, ax=axes[1], palette=['blue', 'red'])
axes[1].set_title('Extracted Space: PC1 vs PC2', fontsize=12)
axes[1].get_legend().remove()

# Plot 3: Constructed Data Space (Risk vs Avg Charges)
sns.scatterplot(x=X_train_constructed['contract_tenure_risk'], y=X_train_constructed['avg_charges_per_service'], hue=y_train, alpha=0.5, ax=axes[2], palette=['blue', 'red'])
axes[2].set_title('Constructed Space: Risk vs Spending Intensity', fontsize=12)
axes[2].legend(title='Churn', loc='upper right')

plt.suptitle('Evolution of the Feature Space', fontsize=16, y=1.05)
plt.tight_layout()
plt.show()

## 13. Application Summary and Workflow Summary

1. **Identify and Preprocess:** Separate numeric and categorical columns. Encode strings to integers. 
2. **Strict Scaling:** PCA requires `StandardScaler` to function correctly. Never skip this step.
3. **Extract (PCA):** Use PCA to distill high-dimensional noise into dense, uncorrelated principal components. This accelerates training and enables 2D visual mapping of complex datasets.
4. **Construct (Domain Knowledge):** Invent new metrics that align with business logic (e.g., tenure risk, spending intensity). This translates human intuition into mathematical signals that algorithms can ingest.
5. **Validation:** Always validate engineered features. Handle zero-division gracefully (`np.inf` to `NaN` to `0`), and confirm that your new features genuinely increase cross-validated model accuracy.

In [ ]:
print("-----------------------------------------------------------")
print("Notebook Execution Complete.")
print("You have successfully implemented Feature Extraction and Construction!")
print("-----------------------------------------------------------")